# 📥 Data & Library Import

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.tools
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn import metrics, svm

pd.set_option('display.max_columns', 100)

In [ ]:
expectancy_data = pd.read_csv('../Life Expectancy Data.csv')


expectancy_data.columns = [c.strip().replace(' ', '_').replace('/', '_') for c in expectancy_data.columns]

expectancy_data.shape

In [ ]:
def calculate_vif(X, thresh=5.0):
    """Drops the highest-VIF feature repeatedly until all remaining VIFs are under the threshold."""
    variables = list(range(X.shape[1]))
    dropped = True
    while dropped:
        dropped = False
        vif = [variance_inflation_factor(X.iloc[:, variables].values, ix)
               for ix in range(X.iloc[:, variables].shape[1])]
        maxloc = vif.index(max(vif))
        if max(vif) > thresh:
            print('dropping \'' + X.iloc[:, variables].columns[maxloc] +
                  '\' at index: ' + str(maxloc))
            del variables[maxloc]
            dropped = True
    print('Remaining variables:')
    print(X.columns[variables])
    return X.iloc[:, variables]

In [ ]:
def stepwise_selection(X, y, threshold_in = 0.01, threshold_out = 0.05, verbose = True):
    # The function is checking for p-values (whether features are statistically significant) - lower is better
    included = [] # this is going to be the list of features we keep
    while True:
        changed = False
        # forward step
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index = excluded, dtype = 'float64')
        for new_column in excluded:
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included + [new_column]]))).fit()
            new_pval[new_column] = model.pvalues[new_column]
        best_pval = new_pval.min()
        # we add the feature with the lowest (best) p-value under the threshold to our 'included' list
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed = True
            if verbose:
                print('Add  {:30} with p-value {:.6}'.format(best_feature, best_pval)) # specifying the verbose text


        # backward step: removing features if new features added to the list make them statistically insignificant
        model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included]))).fit()
        
        # use all coefs except intercept
        pvalues = model.pvalues.iloc[1:]
        worst_pval = pvalues.max() # null if pvalues is empty
        # if the p-value exceeds the upper threshold, the feature will be dropped from the 'included' list
        if worst_pval > threshold_out:
            changed = True
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
            if verbose:
                print('Drop {:30} with p-value {:.6}'.format(worst_feature, worst_pval))
        if not changed:
            break
    return included

In [ ]:
def drop(df:pd.DataFrame, column:str, inplace:bool=False):
    if column in df.columns:
        return df.drop(column, axis=1, inplace=inplace)
    elif not inplace:
        return df

In [ ]:
def standardize(arr):
    return (arr - np.mean(arr)) / np.std(arr)

def normalize(arr):
    return (arr - np.min(arr)) / (np.max(arr) - np.min(arr))

---
# 🔎 Explore Data

In [ ]:
expectancy_data.columns

In [ ]:
expectancy_data['Region'].unique()

In [ ]:
expectancy_data['Country'].unique()


### Data overview

In [ ]:
# TODO: get an overview. head(), dtypes, describe(), shape.
display(expectancy_data.head())
print("\n--- Data Types ---")
print(expectancy_data.dtypes)
display(expectancy_data.describe())
print(f"\nShape: {expectancy_data.shape}")

# Look specifically at which columns are objects and which are numeric
object_cols = expectancy_data.select_dtypes(include=['object']).columns.tolist()
numeric_cols = expectancy_data.select_dtypes(exclude=['object']).columns.tolist()

print(f"\nObject columns: {object_cols}")
print(f"Numeric columns: {numeric_cols}")

### Correlation

In [ ]:
# TODO: look at the distribution of Life Expectancy and at the strongest correlations with it.
sns.histplot(expectancy_data['Life_expectancy'], kde=True)
plt.title('Distribution of LifeExpectancy')
plt.show()

# Calculate correlations for numeric columns only
numeric_df = expectancy_data.select_dtypes(include=[np.number])
correlations = numeric_df.corr()['Life_expectancy'].sort_values(ascending=False)
print("\nStrongest correlations with LifeExpectancy:\n", correlations.head(10))
print("\nWeakest correlations with LifeExpectancy:\n", correlations.tail(5))

In [ ]:
# sns.pairplot(expectancy_data)
# plt.show()

### Check Nulls

In [ ]:
# TODO: count the nulls in every column and show only the columns that have any.
nulls = expectancy_data.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False))

In [ ]:
sns.scatterplot(
    x=np.log(expectancy_data['Schooling']), y=expectancy_data['Life_expectancy']
);

---
# 🧹 Data Preparation

### One Hot Encoding Regions

In [ ]:
df = pd.get_dummies(expectancy_data, columns=['Region'], drop_first=True)

In [ ]:
region_columns = [col for col in df.columns if col.startswith('Region_')]

### Split Data
... into least information and elaborate datasets.

In [ ]:
# Model 1: Least Information (Privacy-Preserving)
least_information_columns = [
    "Life_expectancy",
    "Year",
    "Economy_status_Developed",
    "Economy_status_Developing",
    "Population_mln",
    "GDP_per_capita",
    "Schooling",
    
    # "Adult_mortality",
    # 'Under_five_deaths',
    # 'Incidents_HIV'
]

# Model 2: Elaborate (All Features)
# This includes the base features above, plus all sensitive medical/health records.
elaborate_columns = [
    "Life_expectancy",
    "Year",
    "Economy_status_Developed",
    "Economy_status_Developing",
    "Population_mln",
    "GDP_per_capita",
    "Schooling",
    "Infant_deaths",
    "Under_five_deaths",
    "Adult_mortality",
    "Alcohol_consumption",
    "Hepatitis_B",
    "Measles",
    "BMI",
    "Polio",
    "Diphtheria",
    "Incidents_HIV",
    "Thinness_ten_nineteen_years",
    "Thinness_five_nine_years"
]

In [ ]:
least_information_columns = least_information_columns + region_columns
elaborate_columns = elaborate_columns + region_columns

In [ ]:
df_least = df[least_information_columns]
df_elaborate = df[elaborate_columns]

In [ ]:
df_least.head()

In [ ]:
# df_least['GDP_per_capita_log'] = np.log(df_least['GDP_per_capita'])

In [ ]:
sns.heatmap(
    df_least.corr(numeric_only=True)[['Life_expectancy']].sort_values('Life_expectancy'),
    annot=True
)

plt.show()

In [ ]:
# sns.pairplot(df_least[df_least.columns[::-1]])
# plt.show()

In [ ]:
taget_col = 'Schooling'

sns.scatterplot(
    x=np.log(df_least[taget_col]), y=df_least['Life_expectancy']
)
plt.show()

corr = np.corrcoef(np.log(df_least[taget_col]),df_least['Life_expectancy'])
print(f'Correlation: {corr[0,-1]}')

---
# 📑 Train-Test Split

In [ ]:
def get_train_test(df, test_split:float=0.2, random_state=42, verbose:bool=False):
    # Fetch feature columns, and prepare features & labels
    feature_cols = list(df.columns)
    feature_cols.remove('Life_expectancy')

    X = df[feature_cols].astype(float)
    y = df['Life_expectancy']

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_split, random_state=random_state)

    if verbose:
        print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
        print(f' X_test: {X_test.shape}  |  y_test: {y_test.shape}')

    return X_train, X_test, y_train, y_test

---
# 🤖 Modelling

In [ ]:
def fe_basic(df:pd.DataFrame):
    df['GDP_per_capita_log'] = np.log(df['GDP_per_capita'])

    drop(df, 'Economy_status_Developing', inplace=True)
    # drop(df, 'Economy_status_Developed', inplace=True)
    drop(df, 'GDP_per_capita', inplace=True)
    return df

## Full-Features

In [ ]:
X_train, X_test, y_train, y_test = get_train_test(df_least)
X_train, X_test = fe_basic(X_train), fe_basic(X_test)

In [ ]:
linreg = sm.OLS(y_train, X_train)
results = linreg.fit()
results.summary()

In [ ]:
y_train_pred = results.predict(X_train)
y_test_pred = results.predict(X_test)

full_train_rmse = sm.tools.eval_measures.rmse(y_train, y_train_pred)
full_test_rmse = sm.tools.eval_measures.rmse(y_test, y_test_pred)

print(f'Full-features train RMSE: {full_train_rmse.round(3)}')
print(f'Full-features test RMSE:  {full_test_rmse.round(3)}')

## VIF-Features

In [ ]:
# Calculating VIF
vif_feature_cols = calculate_vif(X_train).columns

In [ ]:
X_train_vif, X_test_vif = X_train[vif_feature_cols], X_test[vif_feature_cols]

linreg = sm.OLS(y_train, X_train_vif)
results = linreg.fit()
results.summary()

In [ ]:
y_train_pred = results.predict(X_train_vif)
y_test_pred = results.predict(X_test_vif)

vif_train_rmse = sm.tools.eval_measures.rmse(y_train, y_train_pred)
vif_test_rmse = sm.tools.eval_measures.rmse(y_test, y_test_pred)

print(f'VIF-features train RMSE: {vif_train_rmse.round(3)}')
print(f'VIF-features test RMSE: {vif_test_rmse.round(3)}')

## Stepwise-Features

In [ ]:
def fe_stepwise(df:pd.DataFrame):
    df['GDP_per_capita_log'] = np.log(df['GDP_per_capita'])
    # drop(df, 'GDP_per_capita', inplace=True)
    df['Schooling_log'] = np.log(df['Schooling'])
    # drop(df, 'Schooling', inplace=True)

    df = sm.add_constant(df)
    return df

In [ ]:
X_train, X_test, y_train, y_test = get_train_test(df_least, random_state=42)
X_train, X_test = fe_stepwise(X_train), fe_stepwise(X_test)

sw_feature_cols = stepwise_selection(X_train, y_train, verbose=False)
X_train, X_test = X_train[sw_feature_cols], X_test[sw_feature_cols]

print(X_train.columns)
display(X_train.head())

linreg = sm.OLS(y_train, X_train)
results = linreg.fit()
results.summary()

# results = RidgeCV()     # 4.203 – 4.506
# results = LassoCV()     # 4.251 – 4.536
# results.fit(X_train, y_train)
# print(results.score(X_train, y_train))

In [ ]:
y_train_pred = results.predict(X_train)
y_test_pred = results.predict(X_test)

sw_train_rmse = sm.tools.eval_measures.rmse(y_train, y_train_pred)
sw_test_rmse = sm.tools.eval_measures.rmse(y_test, y_test_pred)

print(f'SW-features train RMSE: {sw_train_rmse.round(3)}')
print(f'SW-features test RMSE: {sw_test_rmse.round(3)}')

## Coarse-Features
Banding features where an exact value is replaced with a range/band.

In [ ]:
least_information_columns = [
    "Life_expectancy",
    
    "Year",
    # "Economy_status_Developed",
    # "Economy_status_Developing",
    "Population_mln",
    "GDP_per_capita",
    "Schooling",
    
    "Adult_mortality",
    'Under_five_deaths',
    'Incidents_HIV'
]

In [ ]:
bands = 10


In [ ]:
def get_bands(ds:pd.Series, bands:int=10):
    # edges = list(ds.quantile([0,.25,.5,.75,1]).values)
    # edges = list(ds.quantile([0,.2,.4,.6,.8,1]).values)
    edges = list(ds.quantile([x/bands for x in range(bands+1)]).values)
    edges[0], edges[-1] = -np.inf, np.inf
    return edges

def coarsen_col(ds:pd.Series):
    edges = get_bands(ds)
    ds = pd.cut(ds, edges, labels=False)
    

def fe_coarse(df):
    # Transform GDP/capita to log scale
    df['GDP_per_capita_log'] = np.log(df['GDP_per_capita'])
    drop(df, 'GDP_per_capita', inplace=True)
    # df['Schooling_log'] = np.log(df['Schooling'])
    # drop(df, 'Schooling', inplace=True)

    # Coarsen columns to hide sensitive data
    dxt = get_train_test(df_least)[0] # dummy_X_train
    columns = ['Adult_mortality', 'Under_five_deaths', 'Incidents_HIV']

    for col in columns:
        edges = get_bands(dxt[col])
        print(f'{col}: {edges}')
        df[col+''] = pd.cut(df[col], edges, labels=False)

    df = sm.add_constant(df)
    return df


In [ ]:
df_least = df[least_information_columns + region_columns]

X_train, X_test, y_train, y_test = get_train_test(df_least)
X_train, X_test = fe_coarse(X_train), fe_coarse(X_test)

# sw_feature_cols = variance_inflation_factor(X_train, y_train, verbose=True)
# X_train, X_test = X_train[sw_feature_cols], X_test[sw_feature_cols]

display(X_train.head())

linreg = sm.OLS(y_train, X_train)
results = linreg.fit()
results.summary()

# results = RidgeCV()
# results.fit(X_train, y_train)
# print(results.score(X_train, y_train))

In [ ]:
y_train_pred = results.predict(X_train)
y_test_pred = results.predict(X_test)

co_train_rmse = sm.tools.eval_measures.rmse(y_train, y_train_pred)
co_test_rmse = sm.tools.eval_measures.rmse(y_test, y_test_pred)

print(f'Coarse-features train RMSE: {co_train_rmse.round(3)}')
print(f'Coarse-features test RMSE: {co_test_rmse.round(3)}')

## Normalize-Features

In [ ]:
least_information_columns = [
    "Life_expectancy",
    "Year",
    # "Economy_status_Developed",
    # "Economy_status_Developing",
    "Population_mln",
    "GDP_per_capita",
    "Schooling",
    
    "Adult_mortality",
    'Under_five_deaths',
    'Incidents_HIV'
]

In [ ]:
def fe_normalize(df):
    df = fe_coarse(df)

    df['Population_mln'] = standardize(df['Population_mln'])
    df['Schooling'] = standardize(df['Schooling'])

    return df

In [ ]:
df_least = df[least_information_columns + region_columns]

X_train, X_test, y_train, y_test = get_train_test(df_least)
X_train, X_test = fe_normalize(X_train), fe_normalize(X_test)

print(X_train.columns)
display(X_train.head())

linreg = sm.OLS(y_train, X_train)
results = linreg.fit()
results.summary()

In [ ]:
y_train_pred = results.predict(X_train)
y_test_pred = results.predict(X_test)

no_train_rmse = sm.tools.eval_measures.rmse(y_train, y_train_pred)
no_test_rmse = sm.tools.eval_measures.rmse(y_test, y_test_pred)

print(f'Normalize-features train RMSE: {no_train_rmse.round(3)}')
print(f'Normalize-features test RMSE: {no_test_rmse.round(3)}')

# 🏁 RMSE Rankings

In [ ]:
print(f"Full Features:\n\t{full_train_rmse.round(3)} – {full_test_rmse.round(3)}", end='\n\n')
print(f"VIF Features:\n\t{vif_train_rmse.round(3)} – {vif_test_rmse.round(3)}", end='\n\n')
print(f"Stepwise Features:\n\t{sw_train_rmse.round(3)} – {sw_test_rmse.round(3)}", end='\n\n')
print(f"Coarse Features:\n\t{co_train_rmse.round(3)} – {co_test_rmse.round(3)}", end='\n\n')
print(f"Normalized Features:\n\t{no_train_rmse.round(3)} – {no_test_rmse.round(3)}", end='\n\n')

---
# `models.py` Testing

In [ ]:
import models

In [ ]:
mode = 'coarse'

fit, apply = models.MODELS[mode]['fit'], models.MODELS[mode]['apply']
features = models.MODELS[mode]['features']

X = df.drop(columns=["Life_expectancy"])     # features
y = df["Life_expectancy"]                    # the answers
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

state = fit(X_train) # <- YOUR fit, called once

train_fe = sm.add_constant(apply(X_train, state)[features])   # <- YOUR apply

test_fe  = sm.add_constant(apply(X_test,  state)[features])   # <- YOUR apply

model = sm.OLS(y_train, train_fe).fit() # <<<<<< MODEL IS TRAINED HERE
test_rmse = sm.tools.eval_measures.rmse(y_test, model.predict(test_fe))

test_rmse